# Procesar Data y Almacenar en GCS

In [22]:
from google.colab import auth
auth.authenticate_user()

Extraemos data ficticia de una API

In [9]:
import requests
size = '2'
URL = 'https://randomuser.me/api/?results=' + size
response = requests.get(URL)
# response
data = response.json()
data['results']

[{'gender': 'female',
  'name': {'title': 'Mrs', 'first': 'Victória', 'last': 'Ferreira'},
  'location': {'street': {'number': 822, 'name': 'Avenida Vinícius de Morais'},
   'city': 'Guarapari',
   'state': 'Maranhão',
   'country': 'Brazil',
   'postcode': 61844,
   'coordinates': {'latitude': '-78.9321', 'longitude': '112.1675'},
   'timezone': {'offset': '-3:00',
    'description': 'Brazil, Buenos Aires, Georgetown'}},
  'email': 'victoria.ferreira@example.com',
  'login': {'uuid': 'f6651055-94bd-48e5-ad48-64aa440adfbc',
   'username': 'lazygorilla370',
   'password': 'dogface',
   'salt': 'rQZI3uOZ',
   'md5': '29fbf81fd240d4e45cca41ce90531c61',
   'sha1': '5546ec1807e9035a2cef5bc5ee31d283bd4c79f7',
   'sha256': 'fe31ad9bc761f814b17bb6ab2e2e9eba01b1b2b37b518d1c385a0e90647483be'},
  'dob': {'date': '1982-10-20T03:48:01.391Z', 'age': 42},
  'registered': {'date': '2014-11-24T02:04:40.454Z', 'age': 10},
  'phone': '(20) 6897-7659',
  'cell': '(14) 4774-1824',
  'id': {'name': 'CPF', '

Pasamos a DataFrame

In [12]:
import pandas as pd

df = pd.DataFrame(data['results'])
df

,gender,name,location,email,login,dob,registered,phone,cell,id,picture,nat
0,female,"{'title': 'Mrs', 'first': 'Victória', 'last': ...","{'street': {'number': 822, 'name': 'Avenida Vi...",victoria.ferreira@example.com,{'uuid': 'f6651055-94bd-48e5-ad48-64aa440adfbc...,"{'date': '1982-10-20T03:48:01.391Z', 'age': 42}","{'date': '2014-11-24T02:04:40.454Z', 'age': 10}",(20) 6897-7659,(14) 4774-1824,"{'name': 'CPF', 'value': '655.127.197-28'}",{'large': 'https://randomuser.me/api/portraits...,BR
1,male,"{'title': 'Mr', 'first': 'Batista', 'last': 'M...","{'street': {'number': 5097, 'name': 'Rua Santo...",batista.monteiro@example.com,{'uuid': 'f177bb62-d37a-4991-bc3f-c4d062938b32...,"{'date': '1977-04-24T11:59:15.922Z', 'age': 48}","{'date': '2004-06-21T18:21:09.713Z', 'age': 20}",(11) 9578-9310,(20) 3805-0300,"{'name': 'CPF', 'value': '970.792.747-38'}",{'large': 'https://randomuser.me/api/portraits...,BR


In [19]:
rows = []
for index, row in df.iterrows():
    values = {
        'id':row['id']['value'],
        'first_name':row['name']['first'],
        'last_name':row['name']['last'],
        'username':row['login']['username'],
        'email':row['email'],
        'phone_number':row['cell'],
        'date_of_birth':row['dob']['date'],
        'employment_title': row['name']['title'],
        'address_street': row['location']['street']['name'] + ' ' + str(row['location']['street']['number']),
        'address_city': row['location']['city'],
        'address_state': row['location']['state'],
        'address_country': row['location']['country']
    }
    rows.append(values)

df_processed = pd.DataFrame(rows)
df_processed

,id,first_name,last_name,username,email,phone_number,date_of_birth,employment_title,address_street,address_city,address_state,address_country
0,655.127.197-28,Victória,Ferreira,lazygorilla370,victoria.ferreira@example.com,(14) 4774-1824,1982-10-20T03:48:01.391Z,Mrs,Avenida Vinícius de Morais 822,Guarapari,Maranhão,Brazil
1,970.792.747-38,Batista,Monteiro,smallpeacock515,batista.monteiro@example.com,(20) 3805-0300,1977-04-24T11:59:15.922Z,Mr,Rua Santos Dumont 5097,Castanhal,Bahia,Brazil


In [20]:
df.columns

Index(['gender', 'name', 'location', 'email', 'login', 'dob', 'registered',
       'phone', 'cell', 'id', 'picture', 'nat'],
      dtype='object')

In [21]:
df_processed.columns

Index(['id', 'first_name', 'last_name', 'username', 'email', 'phone_number',
       'date_of_birth', 'employment_title', 'address_street', 'address_city',
       'address_state', 'address_country'],
      dtype='object')

## Función para ingesta

In [23]:
from datetime import datetime
import uuid
import random
from google.cloud import storage
import json
import requests

project_id = 'lustrous-setup-457000-h5'
bucket_name = 'sd-gcolab-bucket-01'

def function_ingest_data():
    date_str = datetime.now().strftime('%Y%m%d')
    uuid_4dig = str(uuid.uuid4().hex)[:4]

    destination_blob_name = f'ingested/json/web_users/{date_str}/web_users_{uuid_4dig}.json'

    size = str(random.randint(3,10))
    URL = 'https://randomuser.me/api/?results=' + size
    response = requests.get(URL)
    data = response.json()
    print(data['results'])
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    data_str = json.dumps(data['results'])
    blob.upload_from_string(data_str)
    print(destination_blob_name)

    message = f'Object created at {bucket_name} with name "{destination_blob_name}".'
    print(message)
    return json.dumps({"message": message})

In [24]:
function_ingest_data()

[{'gender': 'female', 'name': {'title': 'Miss', 'first': 'آیناز', 'last': 'موسوی'}, 'location': {'street': {'number': 8950, 'name': 'شهید بهشتی'}, 'city': 'نیشابور', 'state': 'گلستان', 'country': 'Iran', 'postcode': 19322, 'coordinates': {'latitude': '18.3830', 'longitude': '177.1189'}, 'timezone': {'offset': '-9:00', 'description': 'Alaska'}}, 'email': 'aynz.mwswy@example.com', 'login': {'uuid': '2a01e6ce-1d24-4264-ae85-b513ea6c3fe7', 'username': 'greenpeacock387', 'password': 'leng', 'salt': '1lsjArP5', 'md5': 'ec23e7e55268aa4555014d6599e1215e', 'sha1': '0b782a3eab9ca73119f07f5282e2451ddbb5f72c', 'sha256': '493817b4e7753ff0a82509bba0937c0e63f6032790eaff4a1765d3f442c53c8e'}, 'dob': {'date': '1948-11-02T14:53:30.070Z', 'age': 76}, 'registered': {'date': '2006-12-19T06:08:09.789Z', 'age': 18}, 'phone': '064-82994884', 'cell': '0917-727-0077', 'id': {'name': '', 'value': None}, 'picture': {'large': 'https://randomuser.me/api/portraits/women/89.jpg', 'medium': 'https://randomuser.me/api/p

'{"message": "Object created at sd-gcolab-bucket-01 with name \\"ingested/json/web_users/20250526/web_users_e6f7.json\\"."}'

## Función para procesamiento

In [25]:
import pandas as pd
import json
from google.cloud import storage
from datetime import datetime

project_id = 'lustrous-setup-457000-h5'
bucket_name = 'sd-gcolab-bucket-01'

def function_transform_data():

    client = storage.Client()
    bucket = client.get_bucket(bucket_name)
    date_str = datetime.now().strftime('%Y%m%d')
    prefix_source = 'ingested/json/web_users/' + date_str
    prefix_target = 'transformed/csv/web_users/' + date_str
    blobs = bucket.list_blobs(prefix=prefix_source)
    blob_names = [blob.name.split("/")[-1] for blob in blobs]
    print(blob_names)

    rows = []

    for blob in blob_names:
        source_path = 'gs://' + bucket_name + '/' + prefix_source + '/' + blob
        print(source_path)
        target_path = 'gs://' + bucket_name + '/' + prefix_target + '/' + 'web_users.csv'
        print(target_path)
        df = pd.read_json(source_path)
        print(df)

        for index, row in df.iterrows():
            values = {
                'id':row['id']['value'],
                'first_name':row['name']['first'],
                'last_name':row['name']['last'],
                'username':row['login']['username'],
                'email':row['email'],
                'phone_number':row['cell'],
                'date_of_birth':row['dob']['date'],
                'employment_title': row['name']['title'],
                'address_street': row['location']['street']['name'] + ' ' + str(row['location']['street']['number']),
                'address_city': row['location']['city'],
                'address_state': row['location']['state'],
                'address_country': row['location']['country'],
            }
            rows.append(values)

    df_silver = pd.DataFrame(rows)
    print(df_silver)

    message = f'Objects transformed: {source_path} and loaded as: {target_path}'
    print(message)
    df_silver.to_csv(target_path, index=False)

In [26]:
function_transform_data()

['web_users_e6f7.json']
gs://sd-gcolab-bucket-01/ingested/json/web_users/20250526/web_users_e6f7.json
gs://sd-gcolab-bucket-01/transformed/csv/web_users/20250526/web_users.csv
   gender                                               name  \
0  female  {'title': 'Miss', 'first': 'آیناز', 'last': 'م...   
1    male  {'title': 'Mr', 'first': 'Howard', 'last': 'Mi...   
2    male  {'title': 'Mr', 'first': 'Joris', 'last': 'Den...   

                                            location  \
0  {'street': {'number': 8950, 'name': 'شهید بهشت...   
1  {'street': {'number': 3957, 'name': 'York Road...   
2  {'street': {'number': 4928, 'name': 'Rue de Ge...   

                         email  \
0       aynz.mwswy@example.com   
1  howard.mitchell@example.com   
2      joris.denis@example.com   

                                               login  \
0  {'uuid': '2a01e6ce-1d24-4264-ae85-b513ea6c3fe7...   
1  {'uuid': '72bc3b01-bc71-469c-afea-1896ae78095a...   
2  {'uuid': '79074c3b-3951-423c-a3d6-

In [ ]:
function_ingest_data()
function_transform_data()